# Capítulo 10 — Conexões externas, intraconexões e uniões: Novas manobras

**Livro:** começa na página **375** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(10, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## É uma questão de esquerda e direita · PDF p. 376

#### Exercício: Trocando o lado

Agora `brinquedos LEFT JOIN garotos`: quais brinquedos ninguém tem? (Filtre com `g.garoto IS NULL`.)

In [ ]:
%%sql
SELECT b.brinquedo
FROM brinquedos b LEFT JOIN garotos g ON g.brinquedo_id = b.brinquedo_id
WHERE g.garoto IS NULL;

## Aqui está uma conexão externa esquerda · PDF p. 377

#### Exercício: LEFT JOIN

Mostre **todos** os garotos com o brinquedo de cada um. O `Noé` não tem brinquedo: o que aparece?

In [ ]:
%%sql
SELECT g.garoto, b.brinquedo
FROM garotos g LEFT JOIN brinquedos b ON g.brinquedo_id = b.brinquedo_id
ORDER BY g.garoto;

## Conexões externas e combinações múltiplas · PDF p. 382

#### Exercício: LEFT JOIN com várias tabelas

Mostre todos os contatos, o emprego atual (se houver) e o cargo desejado (se houver).

In [ ]:
%%sql
SELECT mc.primeiro_nome, ea.cargo AS cargo_atual, ed.cargo AS cargo_desejado
FROM meus_contatos mc
LEFT JOIN emprego_atual ea ON ea.contato_id = mc.contato_id
LEFT JOIN emprego_desejado ed ON ed.contato_id = mc.contato_id
ORDER BY mc.contato_id;

## Conexão externa direita · PDF p. 383

#### Exercício: RIGHT JOIN

Escreva o mesmo resultado com `garotos RIGHT JOIN brinquedos`.

In [ ]:
%%sql
SELECT b.brinquedo
FROM garotos g RIGHT JOIN brinquedos b ON g.brinquedo_id = b.brinquedo_id
WHERE g.garoto IS NULL;

## Enquanto você estava conectando externamente... · PDF p. 384

#### Exercício: FULL OUTER JOIN

O MySQL do livro não tem `FULL OUTER JOIN`; o PostgreSQL tem. Mostre garotos e brinquedos, incluindo garotos sem brinquedo **e** brinquedos sem garoto.

In [ ]:
%%sql
SELECT g.garoto, b.brinquedo
FROM garotos g FULL OUTER JOIN brinquedos b ON g.brinquedo_id = b.brinquedo_id
ORDER BY g.garoto NULLS LAST, b.brinquedo;

## Uma chave externa auto-referenciada · PDF p. 386

#### Exercício: Chave que aponta para a própria tabela

Em `palhacos`, `chefe_id` aponta para outro palhaço. Insira o palhaço `Bolinha` (id 7) com chefe `Zazá` (id 6) e mostre a tabela.

In [ ]:
%%sql
INSERT INTO palhacos VALUES (7, 'Bolinha', 6);
SELECT * FROM palhacos ORDER BY palhaco_id;

## Nós precisamos de uma autoconexão (SELF-JOIN) · PDF p. 388

#### Exercício: SELF-JOIN

Mostre cada palhaço e o nome do seu chefe juntando `palhacos` com ela mesma (apelidos `p` e `chefe`). Use `LEFT JOIN` para o Pipoca (sem chefe) aparecer.

In [ ]:
%%sql
SELECT p.nome AS palhaco, chefe.nome AS chefe
FROM palhacos p LEFT JOIN palhacos chefe ON p.chefe_id = chefe.palhaco_id
ORDER BY p.palhaco_id;

## Você pode utilizar uma UNION (UNIÃO) · PDF p. 390

#### Exercício: UNION

Liste todos os cargos que aparecem em `emprego_atual`, `emprego_desejado` e `vagas`, sem repetir, em ordem.

In [ ]:
%%sql
SELECT cargo FROM emprego_atual
UNION SELECT cargo FROM emprego_desejado
UNION SELECT cargo FROM vagas
ORDER BY cargo;

## UNION ALL · PDF p. 392

#### Exercício: UNION ALL mantém os repetidos

Conte quantas linhas saem com `UNION` e com `UNION ALL` (use cada união como subconsulta no `FROM`).

In [ ]:
%%sql
SELECT
  (SELECT COUNT(*) FROM (SELECT cargo FROM emprego_atual UNION SELECT cargo FROM emprego_desejado
                         UNION SELECT cargo FROM vagas) AS u) AS com_union,
  (SELECT COUNT(*) FROM (SELECT cargo FROM emprego_atual UNION ALL SELECT cargo FROM emprego_desejado
                         UNION ALL SELECT cargo FROM vagas) AS ua) AS com_union_all;

## Crie uma tabela a partir da sua UNION · PDF p. 393

#### Exercício: CREATE TABLE AS com UNION

Crie a tabela `cargos` a partir da `UNION` dos três cargos.

In [ ]:
%%sql
CREATE TABLE cargos AS
SELECT cargo FROM emprego_atual UNION SELECT cargo FROM emprego_desejado UNION SELECT cargo FROM vagas;
SELECT COUNT(*) AS cargos_distintos FROM cargos;

## INTERSECT e EXCEPT · PDF p. 393

#### Exercício: INTERSECT e EXCEPT

Quais cargos têm vaga aberta **e** alguém que deseja (`INTERSECT`)? E quais cargos têm vaga mas ninguém deseja (`EXCEPT`)? (O MySQL antigo do livro não tinha esses dois.) Mostre os dois com uma coluna de identificação.

In [ ]:
%%sql
(SELECT 'vaga e desejo' AS grupo, cargo FROM vagas INTERSECT SELECT 'vaga e desejo', cargo FROM emprego_desejado)
UNION ALL
(SELECT 'só vaga', cargo FROM vagas EXCEPT SELECT 'só vaga', cargo FROM emprego_desejado)
ORDER BY grupo, cargo;

## Transformando uma subconsulta em uma conexão · PDF p. 395

#### Exercício: De subconsulta para junção

Reescreva com `INNER JOIN` a consulta do capítulo 9: quem deseja um cargo que aparece em `vagas`. Use `DISTINCT` para não repetir.

In [ ]:
%%sql
SELECT DISTINCT mc.primeiro_nome, ed.cargo
FROM meus_contatos mc
JOIN emprego_desejado ed ON ed.contato_id = mc.contato_id
JOIN vagas v ON v.cargo = ed.cargo
ORDER BY mc.primeiro_nome;

## Um autorrelacionamento como uma subconsulta · PDF p. 399

#### Exercício: Self-join com subconsulta

Mostre cada palhaço e o nome do chefe usando uma subconsulta no `SELECT` em vez de junção.

In [ ]:
%%sql
SELECT p.nome AS palhaco,
       (SELECT c.nome FROM palhacos c WHERE c.palhaco_id = p.chefe_id) AS chefe
FROM palhacos p ORDER BY p.palhaco_id;